In [ ]:

import torch

print("Versão do PyTorch:", torch.__version__)
print("GPU disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU não disponível")


Versão do PyTorch: 2.14.1+cu130
GPU disponível: True
GPU: Tesla T4


In [ ]:

!pip -q install "transformers>=4.45,<5" accelerate


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:

import transformers
import accelerate
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)
print("Bibliotecas carregadas com sucesso!")


Transformers: 4.57.6
Accelerate: 1.15.0
Hugging Face Hub: 0.36.2
Bibliotecas carregadas com sucesso!


In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

modelo_nome = "Qwen/Qwen2.5-1.5B-Instruct"

print("Baixando e carregando a IA...")

tokenizer = AutoTokenizer.from_pretrained(modelo_nome)

model = AutoModelForCausalLM.from_pretrained(
    modelo_nome,
    dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Modelo carregado com sucesso!")
print("Dispositivo:", model.device)


Baixando e carregando a IA...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Modelo carregado com sucesso!
Dispositivo: cuda:0


In [ ]:

pergunta = "Quem é responsável pela manutenção das calçadas na cidade de São Paulo?"

mensagens = [
    {
        "role": "system",
        "content": "Você é um assistente que responde em português brasileiro de forma clara e objetiva."
    },
    {
        "role": "user",
        "content": pergunta
    }
]

entradas = tokenizer.apply_chat_template(
    mensagens,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    saida = model.generate(
        **entradas,
        max_new_tokens=180,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

resposta = tokenizer.decode(
    saida[0][entradas["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("PERGUNTA:", pergunta)
print("\nRESPOSTA DA IA:", resposta)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


PERGUNTA: Quem é responsável pela manutenção das calçadas na cidade de São Paulo?

RESPOSTA DA IA: A responsabilidade pela manutenção das calçadas na cidade de São Paulo é compartilhada entre várias entidades:

1. Prefeitura Municipal: É a principal responsável por projetar, construir e manter as calçadas da cidade.

2. Secretaria de Obras Públicas: Tem o papel de gerenciar os serviços de infraestrutura urbana, incluindo calçadas.

3. Consórcio Intermunicipal de Transportes Metropolitanos (CITM): Desenvolve programas para melhorias nas áreas metropolitanas.

4. Fundação Cultural São Paulo: Apoiou projetos urbanísticos que envolvem calçadas.

5. Companhias de Saneamento Básico: Responsáveis pelo tratamento do lixo e possivelmente também pela limpeza das calçadas.

6.


In [ ]:

import json

base_conhecimento = [
    {
        "id": "lei_15442_art7",
        "fonte": "Lei 15.442/2011 - Art. 7",
        "url": "https://legislacao.prefeitura.sp.gov.br/lei-15442-de-09-de-setembro-de-2011/consolidado",
        "texto": "Em regra, os responsáveis pelos imóveis devem executar, manter e conservar as calçadas correspondentes à frente do imóvel, observadas as exceções legais."
    },
    {
        "id": "lei_15442_art10",
        "fonte": "Lei 15.442/2011 - Art. 10",
        "url": "https://legislacao.prefeitura.sp.gov.br/lei-15442-de-09-de-setembro-de-2011/consolidado",
        "texto": "Entre os responsáveis pela manutenção estão proprietários, possuidores e condomínios. Órgãos públicos respondem pelos imóveis sob sua responsabilidade."
    },
    {
        "id": "lei_15442_art8",
        "fonte": "Lei 15.442/2011 - Art. 8",
        "url": "https://legislacao.prefeitura.sp.gov.br/lei-15442-de-09-de-setembro-de-2011/consolidado",
        "texto": "A instalação de mobiliário urbano não pode impedir a circulação segura dos pedestres. Deve ser respeitada uma faixa livre mínima de 1,20 metro."
    },
    {
        "id": "lei_15442_art11",
        "fonte": "Lei 15.442/2011 - Art. 11",
        "url": "https://legislacao.prefeitura.sp.gov.br/lei-15442-de-09-de-setembro-de-2011/consolidado",
        "texto": "O prazo previsto para regularização após multa e intimação é de 60 dias, reduzido para 20 dias nos casos das irregularidades do artigo 8."
    },
    {
        "id": "decreto_59671_art34",
        "fonte": "Decreto 59.671/2020 - Art. 34",
        "url": "https://legislacao.prefeitura.sp.gov.br/decreto-59671-de-7-de-agosto-de-2020/consolidado",
        "texto": "Calçadas com buracos, ondulações, desníveis indevidos ou obstáculos que prejudiquem a circulação segura são consideradas em mau estado de conservação."
    }
]

with open("base_conhecimento.json", "w", encoding="utf-8") as arquivo:
    json.dump(base_conhecimento, arquivo, ensure_ascii=False, indent=2)

print("Base criada com sucesso!")
print("Total de trechos:", len(base_conhecimento))

for trecho in base_conhecimento:
    print("-", trecho["id"])



Base criada com sucesso!
Total de trechos: 5
- lei_15442_art7
- lei_15442_art10
- lei_15442_art8
- lei_15442_art11
- decreto_59671_art34


In [ ]:

!pip -q install "sentence-transformers==3.4.1"


In [ ]:

from sentence_transformers import SentenceTransformer
import numpy as np

print("Carregando modelo de embeddings...")

buscador = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="cpu"
)

documentos = [
    trecho["fonte"] + " " + trecho["texto"]
    for trecho in base_conhecimento
]

vetores = buscador.encode(
    documentos,
    convert_to_numpy=True,
    normalize_embeddings=True
)

def buscar_trechos(pergunta, top_k=3):
    vetor_pergunta = buscador.encode(
        [pergunta],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    pontuacoes = vetores @ vetor_pergunta
    indices = np.argsort(-pontuacoes)[:top_k]

    resultados = []

    for i in indices:
        trecho = base_conhecimento[int(i)].copy()
        trecho["similaridade"] = round(
            float(pontuacoes[i]), 3
        )
        resultados.append(trecho)

    return resultados

print("Busca semântica configurada!")
print("Documentos indexados:", len(documentos))


Carregando modelo de embeddings...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Busca semântica configurada!
Documentos indexados: 5


In [ ]:

pergunta = "Quem é responsável pela manutenção das calçadas na cidade de São Paulo?"

resultados = buscar_trechos(pergunta)

print("PERGUNTA:", pergunta)
print("\nTRECHOS ENCONTRADOS:\n")

for posicao, trecho in enumerate(resultados, 1):
    print(f"{posicao}. {trecho['fonte']}")
    print(f"Similaridade: {trecho['similaridade']}")
    print(f"Texto: {trecho['texto']}")
    print("-" * 50)


PERGUNTA: Quem é responsável pela manutenção das calçadas na cidade de São Paulo?

TRECHOS ENCONTRADOS:

1. Lei 15.442/2011 - Art. 7
Similaridade: 0.467
Texto: Em regra, os responsáveis pelos imóveis devem executar, manter e conservar as calçadas correspondentes à frente do imóvel, observadas as exceções legais.
--------------------------------------------------
2. Lei 15.442/2011 - Art. 8
Similaridade: 0.44
Texto: A instalação de mobiliário urbano não pode impedir a circulação segura dos pedestres. Deve ser respeitada uma faixa livre mínima de 1,20 metro.
--------------------------------------------------
3. Lei 15.442/2011 - Art. 10
Similaridade: 0.418
Texto: Entre os responsáveis pela manutenção estão proprietários, possuidores e condomínios. Órgãos públicos respondem pelos imóveis sob sua responsabilidade.
--------------------------------------------------


In [ ]:

import json
import torch

pergunta = "Quem é responsável pela manutenção das calçadas na cidade de São Paulo?"

# Buscar trechos relevantes
trechos = buscar_trechos(pergunta, top_k=3)

# Preparar o contexto da legislação
contexto = "\n\n".join([
    f"Fonte: {t['fonte']}\nTexto: {t['texto']}"
    for t in trechos
])

# Instruções para a IA
mensagens = [
    {
        "role": "system",
        "content": """
Você é o assistente do Passo Firme.

Regras:
1. Responda apenas com base no contexto fornecido.
2. Não invente leis, artigos ou informações.
3. Se não houver informação suficiente, diga que não consta.
4. Cite as fontes utilizadas.
5. Responda exclusivamente em JSON válido, com:
   "resposta": texto,
   "fontes": lista de fontes,
   "recusa": true ou false.
"""
    },
    {
        "role": "user",
        "content": f"CONTEXTO:\n{contexto}\n\nPERGUNTA:\n{pergunta}"
    }
]

entradas = tokenizer.apply_chat_template(
    mensagens,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    saida = model.generate(
        **entradas,
        max_new_tokens=300,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

resposta_rag = tokenizer.decode(
    saida[0][entradas["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("RESPOSTA DA IA COM RAG:\n")
print(resposta_rag)

try:
    json.loads(resposta_rag)
    print("\nJSON válido!")
except json.JSONDecodeError:
    print("\nAtenção: JSON inválido.")


RESPOSTA DA IA COM RAG:

```json
{
  "resposta": "De acordo com a lei citada (Lei 15.442/2011), entre os responsáveis pela manutenção das calçadas na cidade de São Paulo estão proprietários, possuidores e condomínios.",
  "fontes": [
    {
      "url": "http://www.planalto.gov.br/ccivil_03/leis/l15442.htm",
      "descricao": "Link para a lei específica"
    }
  ],
  "recusa": false
}
```

Atenção: JSON inválido.


In [ ]:

import json
import re
import torch

# Usar os trechos encontrados anteriormente
fontes_validas = {t["id"]: t for t in trechos}

contexto = "\n\n".join(
    f"ID: {t['id']}\nFonte: {t['fonte']}\nTexto: {t['texto']}"
    for t in trechos
)

mensagens = [
    {
        "role": "system",
        "content": """
Você é o assistente do Passo Firme.

REGRAS OBRIGATÓRIAS:
1. Responda apenas com informações do contexto.
2. Nunca invente leis, artigos ou links.
3. Use somente os IDs das fontes fornecidas.
4. Se não encontrar a resposta, recuse.
5. Retorne JSON puro, sem markdown.

Formato obrigatório:
{
  "resposta": "texto",
  "fontes": ["id_da_fonte"],
  "recusa": false
}
"""
    },
    {
        "role": "user",
        "content": f"CONTEXTO:\n{contexto}\n\nPERGUNTA:\n{pergunta}"
    }
]

entradas = tokenizer.apply_chat_template(
    mensagens,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    saida = model.generate(
        **entradas,
        max_new_tokens=250,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

texto = tokenizer.decode(
    saida[0][entradas["input_ids"].shape[-1]:],
    skip_special_tokens=True
).strip()

# Remover possíveis marcadores markdown
texto = re.sub(r"^```(?:json)?\s*", "", texto)
texto = re.sub(r"\s*```$", "", texto)

try:
    dados = json.loads(texto)

    estrutura_ok = (
        isinstance(dados, dict)
        and isinstance(dados.get("resposta"), str)
        and isinstance(dados.get("fontes"), list)
        and isinstance(dados.get("recusa"), bool)
        and all(isinstance(f, str) for f in dados["fontes"])
    )

    if not estrutura_ok:
        raise ValueError("Estrutura JSON incorreta")

    if not all(f in fontes_validas for f in dados["fontes"]):
        raise ValueError("A IA citou uma fonte inexistente")

    if "http" in dados["resposta"].lower():
        raise ValueError("A IA inseriu um link na resposta")

    if not dados["recusa"] and not dados["fontes"]:
        raise ValueError("Resposta sem fonte")

    print("JSON válido!")
    print("Fontes verificadas!")
    print(json.dumps(dados, indent=2, ensure_ascii=False))

    print("\nFONTES OFICIAIS:")
    for fonte_id in dados["fontes"]:
        print(fontes_validas[fonte_id]["url"])

except (json.JSONDecodeError, ValueError) as erro:
    print("Falha na validação:", erro)
    print("Resposta original:", texto)


JSON válido!
Fontes verificadas!
{
  "resposta": "De acordo com a Lei nº 15.442/2011, os responsáveis pelas calçadas correspondentes ao imóvel são os proprietários, possuidores e condomínios. Os órgãos públicos são responsáveis por imóveis sob sua responsabilidade.",
  "fontes": [
    "lei_15442_art7",
    "lei_15442_art8",
    "lei_15442_art10"
  ],
  "recusa": false
}

FONTES OFICIAIS:
https://legislacao.prefeitura.sp.gov.br/lei-15442-de-09-de-setembro-de-2011/consolidado
https://legislacao.prefeitura.sp.gov.br/lei-15442-de-09-de-setembro-de-2011/consolidado
https://legislacao.prefeitura.sp.gov.br/lei-15442-de-09-de-setembro-de-2011/consolidado


In [ ]:

# Pergunta cuja resposta não está na base
pergunta_teste = (
    "Quantas reclamações sobre calçadas foram "
    "registradas ontem em São Paulo?"
)

trechos_teste = buscar_trechos(pergunta_teste, top_k=3)

contexto_teste = "\n\n".join(
    f"ID: {t['id']}\nTexto: {t['texto']}"
    for t in trechos_teste
)

mensagens_teste = [
    {
        "role": "system",
        "content": """
Você é o assistente do Passo Firme.
Responda somente com informações explícitas do contexto.
Se não houver dados suficientes, responda "Não consta
na base de conhecimento fornecida".
Não use conhecimento externo ou estimativas.
Nunca invente números ou fontes.
Retorne apenas JSON puro, sem markdown.

Formato:
{
  "resposta": "texto",
  "fontes": [],
  "recusa": true
}

Se a resposta estiver no contexto, use recusa false
e cite somente os IDs dos trechos que a sustentam.
"""
    },
    {
        "role": "user",
        "content": f"CONTEXTO:\n{contexto_teste}\n\nPERGUNTA:\n{pergunta_teste}"
    }
]

entradas = tokenizer.apply_chat_template(
    mensagens_teste,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    saida = model.generate(
        **entradas,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

texto_teste = tokenizer.decode(
    saida[0][entradas["input_ids"].shape[-1]:],
    skip_special_tokens=True
).strip()

print("RESPOSTA DA IA:")
print(texto_teste)


RESPOSTA DA IA:
Não consta na base de conhecimento fornecida.


In [ ]:

import json

resposta_original = texto_teste.strip()

mensagem_recusa = "Não consta na base de conhecimento fornecida."

if resposta_original.lower().rstrip(".") == mensagem_recusa.lower().rstrip("."):
    resultado = {
        "resposta": mensagem_recusa,
        "fontes": [],
        "recusa": True
    }
    formato_original_valido = False
else:
    try:
        resultado = json.loads(resposta_original)
        formato_original_valido = True
    except json.JSONDecodeError:
        resultado = {
            "resposta": "Não foi possível validar a resposta.",
            "fontes": [],
            "recusa": True
        }
        formato_original_valido = False

print("JSON final:")
print(json.dumps(resultado, indent=2, ensure_ascii=False))

print("\nJSON gerado originalmente pela LLM:",
      formato_original_valido)


JSON final:
{
  "resposta": "Não consta na base de conhecimento fornecida.",
  "fontes": [],
  "recusa": true
}

JSON gerado originalmente pela LLM: False


In [ ]:

import json
import os

testes = [
    {
        "pergunta": "Quem deve conservar a calçada em frente ao imóvel?",
        "gabarito": "Os responsáveis pelo imóvel.",
        "fontes_esperadas": ["lei_15442_art7"],
        "deve_recusar": False
    },
    {
        "pergunta": "Quem deve executar e manter as calçadas?",
        "gabarito": "Em regra, os responsáveis pelos imóveis.",
        "fontes_esperadas": ["lei_15442_art7"],
        "deve_recusar": False
    },
    {
        "pergunta": "Condomínios podem ser responsáveis pelas calçadas?",
        "gabarito": "Sim, condomínios estão entre os responsáveis.",
        "fontes_esperadas": ["lei_15442_art10"],
        "deve_recusar": False
    },
    {
        "pergunta": "Órgãos públicos cuidam das calçadas de seus imóveis?",
        "gabarito": "Sim, dos imóveis sob sua responsabilidade.",
        "fontes_esperadas": ["lei_15442_art10"],
        "deve_recusar": False
    },
    {
        "pergunta": "Qual é a largura mínima da faixa livre?",
        "gabarito": "1,20 metro.",
        "fontes_esperadas": ["lei_15442_art8"],
        "deve_recusar": False
    },
    {
        "pergunta": "Mobiliário urbano pode impedir a passagem segura?",
        "gabarito": "Não, deve preservar a circulação segura.",
        "fontes_esperadas": ["lei_15442_art8"],
        "deve_recusar": False
    },
    {
        "pergunta": "Qual é o prazo geral de regularização após intimação?",
        "gabarito": "60 dias, conforme o resumo da base.",
        "fontes_esperadas": ["lei_15442_art11"],
        "deve_recusar": False
    },
    {
        "pergunta": "Qual prazo consta para irregularidades do artigo 8?",
        "gabarito": "20 dias, conforme o resumo da base.",
        "fontes_esperadas": ["lei_15442_art11"],
        "deve_recusar": False
    },
    {
        "pergunta": "Buracos podem caracterizar calçada mal conservada?",
        "gabarito": "Sim, quando prejudicam a circulação segura.",
        "fontes_esperadas": ["decreto_59671_art34"],
        "deve_recusar": False
    },
    {
        "pergunta": "Desníveis indevidos prejudicam a conservação da calçada?",
        "gabarito": "Sim, podem caracterizar mau estado de conservação.",
        "fontes_esperadas": ["decreto_59671_art34"],
        "deve_recusar": False
    }
]

perguntas_sem_resposta = [
    "Quantas reclamações sobre calçadas foram registradas ontem?",
    "Qual é o telefone pessoal do prefeito?",
    "Quantos acidentes ocorreram hoje na Avenida Paulista?",
    "Qual foi o orçamento de calçadas de São Paulo em 2025?",
    "Quantos idosos caíram em calçadas no último mês?"
]

for pergunta in perguntas_sem_resposta:
    testes.append({
        "pergunta": pergunta,
        "gabarito": "Não consta na base de conhecimento fornecida.",
        "fontes_esperadas": [],
        "deve_recusar": True
    })

os.makedirs("avaliacao", exist_ok=True)

with open("avaliacao/testes.json", "w", encoding="utf-8") as arquivo:
    json.dump(testes, arquivo, ensure_ascii=False, indent=2)

print("Conjunto de testes criado!")
print("Total de perguntas:", len(testes))
print("Com resposta:", sum(not t["deve_recusar"] for t in testes))
print("Sem resposta:", sum(t["deve_recusar"] for t in testes))


Conjunto de testes criado!
Total de perguntas: 15
Com resposta: 10
Sem resposta: 5


In [ ]:

import json

resultados_avaliacao = []
acertos_1 = 0
acertos_3 = 0
total_com_resposta = 0

for teste in testes:
    pergunta = teste["pergunta"]
    encontrados = buscar_trechos(pergunta, top_k=3)

    ids = [t["id"] for t in encontrados]
    esperado = set(teste["fontes_esperadas"])

    if teste["deve_recusar"]:
        print(f"SEM RESPOSTA: {pergunta}")
        print(f"Maior similaridade: {encontrados[0]['similaridade']}")
        print("-" * 40)
        continue

    total_com_resposta += 1

    recall1 = len(esperado.intersection(ids[:1])) / len(esperado)
    recall3 = len(esperado.intersection(ids[:3])) / len(esperado)

    acertos_1 += recall1
    acertos_3 += recall3

    resultados_avaliacao.append({
        "pergunta": pergunta,
        "esperado": list(esperado),
        "encontrados": ids,
        "recall_1": recall1,
        "recall_3": recall3
    })

    print(f"PERGUNTA: {pergunta}")
    print(f"Recall@1: {recall1}")
    print(f"Recall@3: {recall3}")
    print("-" * 40)

recall_medio_1 = acertos_1 / total_com_resposta
recall_medio_3 = acertos_3 / total_com_resposta

print("\n===== RESULTADO FINAL =====")
print(f"Recall@1: {recall_medio_1:.2%}")
print(f"Recall@3: {recall_medio_3:.2%}")

with open("avaliacao/recall_resultados.json", "w", encoding="utf-8") as arquivo:
    json.dump({
        "recall_1": recall_medio_1,
        "recall_3": recall_medio_3,
        "resultados": resultados_avaliacao
    }, arquivo, indent=2, ensure_ascii=False)

print("\nResultados salvos!")


PERGUNTA: Quem deve conservar a calçada em frente ao imóvel?
Recall@1: 1.0
Recall@3: 1.0
----------------------------------------
PERGUNTA: Quem deve executar e manter as calçadas?
Recall@1: 1.0
Recall@3: 1.0
----------------------------------------
PERGUNTA: Condomínios podem ser responsáveis pelas calçadas?
Recall@1: 0.0
Recall@3: 0.0
----------------------------------------
PERGUNTA: Órgãos públicos cuidam das calçadas de seus imóveis?
Recall@1: 0.0
Recall@3: 1.0
----------------------------------------
PERGUNTA: Qual é a largura mínima da faixa livre?
Recall@1: 1.0
Recall@3: 1.0
----------------------------------------
PERGUNTA: Mobiliário urbano pode impedir a passagem segura?
Recall@1: 1.0
Recall@3: 1.0
----------------------------------------
PERGUNTA: Qual é o prazo geral de regularização após intimação?
Recall@1: 1.0
Recall@3: 1.0
----------------------------------------
PERGUNTA: Qual prazo consta para irregularidades do artigo 8?
Recall@1: 1.0
Recall@3: 1.0
-----------------

In [ ]:

# Avaliar diferentes limiares de similaridade

resultados_busca = []

for teste in testes:
    encontrados = buscar_trechos(teste["pergunta"], top_k=3)
    resultados_busca.append((teste, encontrados))

limiares = [0.30, 0.35, 0.40, 0.42, 0.45, 0.48, 0.50, 0.55, 0.60]

print("LIMIAR | FONTES CORRETAS | RECUSAS CORRETAS")
print("-" * 49)

for limiar in limiares:
    fontes_corretas = 0
    recusas_corretas = 0

    for teste, encontrados in resultados_busca:
        aprovados = [
            t for t in encontrados
            if t["similaridade"] >= limiar
        ]

        if teste["deve_recusar"]:
            if len(aprovados) == 0:
                recusas_corretas += 1
        else:
            ids = [t["id"] for t in aprovados]

            if any(
                fonte in ids
                for fonte in teste["fontes_esperadas"]
            ):
                fontes_corretas += 1

    print(
        f"{limiar:.2f}   | "
        f"{fontes_corretas}/10            | "
        f"{recusas_corretas}/5"
    )


LIMIAR | FONTES CORRETAS | RECUSAS CORRETAS
-------------------------------------------------
0.30   | 9/10            | 2/5
0.35   | 9/10            | 2/5
0.40   | 8/10            | 4/5
0.42   | 8/10            | 5/5
0.45   | 8/10            | 5/5
0.48   | 8/10            | 5/5
0.50   | 8/10            | 5/5
0.55   | 7/10            | 5/5
0.60   | 7/10            | 5/5


In [ ]:

LIMIAR = 0.42

def buscar_trechos_filtrados(pergunta, top_k=3):
    resultados = buscar_trechos(pergunta, top_k=top_k)

    aprovados = [
        trecho for trecho in resultados
        if trecho["similaridade"] >= LIMIAR
    ]

    return aprovados

# Testar com três perguntas
perguntas = [
    "Quem deve conservar a calçada em frente ao imóvel?",
    "Quantas reclamações sobre calçadas foram registradas ontem?",
    "Condomínios podem ser responsáveis pelas calçadas?"
]

for pergunta in perguntas:
    resultados = buscar_trechos_filtrados(pergunta)

    print("\nPERGUNTA:", pergunta)

    if resultados:
        print("Documentos encontrados:")
        for trecho in resultados:
            print(
                trecho["id"],
                "- Similaridade:",
                trecho["similaridade"]
            )
    else:
        print("Nenhum documento passou pelo limiar.")
        print("Sistema deve recusar ou solicitar revisão.")

    print("-" * 50)



PERGUNTA: Quem deve conservar a calçada em frente ao imóvel?
Documentos encontrados:
lei_15442_art7 - Similaridade: 0.69
lei_15442_art8 - Similaridade: 0.535
lei_15442_art10 - Similaridade: 0.518
--------------------------------------------------

PERGUNTA: Quantas reclamações sobre calçadas foram registradas ontem?
Nenhum documento passou pelo limiar.
Sistema deve recusar ou solicitar revisão.
--------------------------------------------------

PERGUNTA: Condomínios podem ser responsáveis pelas calçadas?
Nenhum documento passou pelo limiar.
Sistema deve recusar ou solicitar revisão.
--------------------------------------------------


In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Criar busca por palavras-chave
tfidf = TfidfVectorizer(
    strip_accents="unicode",
    ngram_range=(1, 2)
)

matriz_tfidf = tfidf.fit_transform(documentos)

def buscar_hibrido(pergunta, top_k=3):

    # Busca semântica
    vetor = buscador.encode(
        [pergunta],
        normalize_embeddings=True
    )[0]

    scores_semanticos = vetores @ vetor

    # Busca por palavras-chave
    vetor_texto = tfidf.transform([pergunta])

    scores_palavras = (
        vetor_texto @ matriz_tfidf.T
    ).toarray().ravel()

    # Combinar os dois métodos
    scores = (
        0.55 * scores_semanticos +
        0.45 * scores_palavras
    )

    indices = np.argsort(-scores)[:top_k]

    resultados = []

    for i in indices:
        trecho = base_conhecimento[int(i)].copy()
        trecho["score_hibrido"] = round(
            float(scores[i]), 3
        )
        resultados.append(trecho)

    return resultados

# Testar novamente
pergunta = "Condomínios podem ser responsáveis pelas calçadas?"

resultados = buscar_hibrido(pergunta)

print("RESULTADOS DA BUSCA HÍBRIDA:\n")

for trecho in resultados:
    print(trecho["id"])
    print("Pontuação:", trecho["score_hibrido"])
    print("-" * 40)


RESULTADOS DA BUSCA HÍBRIDA:

lei_15442_art7
Pontuação: 0.257
----------------------------------------
lei_15442_art8
Pontuação: 0.255
----------------------------------------
decreto_59671_art34
Pontuação: 0.221
----------------------------------------


In [ ]:

pergunta = "Condomínios podem ser responsáveis pelas calçadas?"

vetor_pergunta = buscador.encode(
    [pergunta],
    normalize_embeddings=True
)[0]

scores_sem = vetores @ vetor_pergunta

scores_tfidf = (
    tfidf.transform([pergunta]) @ matriz_tfidf.T
).toarray().ravel()

scores_hibridos = (
    0.55 * scores_sem +
    0.45 * scores_tfidf
)

print("COMPARAÇÃO DOS 5 DOCUMENTOS:\n")

for i in np.argsort(-scores_hibridos):
    print("Documento:", base_conhecimento[int(i)]["id"])
    print("Semântica:", round(float(scores_sem[i]), 3))
    print("Palavras-chave:", round(float(scores_tfidf[i]), 3))
    print("Híbrida:", round(float(scores_hibridos[i]), 3))
    print("-" * 40)


COMPARAÇÃO DOS 5 DOCUMENTOS:

Documento: lei_15442_art7
Semântica: 0.376
Palavras-chave: 0.111
Híbrida: 0.257
----------------------------------------
Documento: lei_15442_art8
Semântica: 0.395
Palavras-chave: 0.084
Híbrida: 0.255
----------------------------------------
Documento: decreto_59671_art34
Semântica: 0.358
Palavras-chave: 0.053
Híbrida: 0.221
----------------------------------------
Documento: lei_15442_art10
Semântica: 0.232
Palavras-chave: 0.15
Híbrida: 0.195
----------------------------------------
Documento: lei_15442_art11
Semântica: 0.084
Palavras-chave: 0.0
Híbrida: 0.046
----------------------------------------


In [ ]:

# Nova versão da busca híbrida

def buscar_hibrido(pergunta, top_k=3):

    vetor = buscador.encode(
        [pergunta],
        normalize_embeddings=True
    )[0]

    scores_sem = vetores @ vetor

    scores_palavras = (
        tfidf.transform([pergunta]) @ matriz_tfidf.T
    ).toarray().ravel()

    scores = (
        0.30 * scores_sem +
        0.70 * scores_palavras
    )

    indices = np.argsort(-scores)[:top_k]

    resultados = []

    for i in indices:
        trecho = base_conhecimento[int(i)].copy()
        trecho["score_hibrido"] = round(float(scores[i]), 3)
        resultados.append(trecho)

    return resultados


# Testar a pergunta problemática
pergunta = "Condomínios podem ser responsáveis pelas calçadas?"

print("RESULTADOS PARA CONDOMÍNIOS:")

for trecho in buscar_hibrido(pergunta):
    print(trecho["id"], trecho["score_hibrido"])


# Reavaliar todas as perguntas com resposta
acertos_1 = 0
acertos_3 = 0
total = 0

for teste in testes:

    if teste["deve_recusar"]:
        continue

    total += 1

    encontrados = buscar_hibrido(teste["pergunta"])
    ids = [t["id"] for t in encontrados]

    esperados = set(teste["fontes_esperadas"])

    if esperados.intersection(ids[:1]):
        acertos_1 += 1

    if esperados.intersection(ids[:3]):
        acertos_3 += 1

print("\nNOVAS MÉTRICAS:")
print(f"Recall@1: {acertos_1 / total:.0%}")
print(f"Recall@3: {acertos_3 / total:.0%}")


RESULTADOS PARA CONDOMÍNIOS:
lei_15442_art7 0.19
lei_15442_art8 0.177
lei_15442_art10 0.175

NOVAS MÉTRICAS:
Recall@1: 90%
Recall@3: 100%


In [ ]:

# Testar limiares para a busca híbrida

limiares = [
    0.10, 0.12, 0.14, 0.16,
    0.17, 0.18, 0.19, 0.20,
    0.22, 0.24, 0.26, 0.28,
    0.30, 0.35
]

print("LIMIAR | FONTES CORRETAS | RECUSAS CORRETAS")
print("-" * 49)

for limiar in limiares:
    acertos = 0
    recusas = 0

    for teste in testes:
        encontrados = buscar_hibrido(
            teste["pergunta"], top_k=3
        )

        aprovados = [
            t for t in encontrados
            if t["score_hibrido"] >= limiar
        ]

        if teste["deve_recusar"]:
            if not aprovados:
                recusas += 1
        else:
            ids = [t["id"] for t in aprovados]
            if any(
                f in ids
                for f in teste["fontes_esperadas"]
            ):
                acertos += 1

    print(
        f"{limiar:.2f}   | "
        f"{acertos}/10            | "
        f"{recusas}/5"
    )


LIMIAR | FONTES CORRETAS | RECUSAS CORRETAS
-------------------------------------------------
0.10   | 10/10            | 0/5
0.12   | 10/10            | 1/5
0.14   | 10/10            | 1/5
0.16   | 10/10            | 2/5
0.17   | 10/10            | 2/5
0.18   | 9/10            | 2/5
0.19   | 9/10            | 2/5
0.20   | 9/10            | 2/5
0.22   | 9/10            | 4/5
0.24   | 9/10            | 4/5
0.26   | 9/10            | 5/5
0.28   | 9/10            | 5/5
0.30   | 8/10            | 5/5
0.35   | 7/10            | 5/5


In [ ]:

import json
import os
import time
import torch

LIMIAR_HIBRIDO = 0.26

os.makedirs("avaliacao", exist_ok=True)

def gerar_resposta(pergunta, contexto=None):
    mensagens = [
        {
            "role": "system",
            "content": """
Você é o assistente do Passo Firme.
Responda em português brasileiro.
Use o contexto quando ele estiver disponível.
Se não houver contexto, utilize seu conhecimento,
mas não invente informações ou referências.
Se não souber, responda que não consta.
Retorne somente JSON puro, sem markdown.

Formato:
{
  "resposta": "texto",
  "fontes": [],
  "recusa": false
}

Se recusar, use recusa true.
Cite apenas IDs de fontes fornecidas no contexto.
"""
        },
        {
            "role": "user",
            "content": (
                f"CONTEXTO:\n{contexto or 'Nenhum'}"
                f"\n\nPERGUNTA:\n{pergunta}"
            )
        }
    ]

    entradas = tokenizer.apply_chat_template(
        mensagens,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    inicio = time.perf_counter()

    with torch.no_grad():
        saida = model.generate(
            **entradas,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    latencia = time.perf_counter() - inicio

    resposta = tokenizer.decode(
        saida[0][entradas["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    return resposta, round(latencia, 2)


# Avaliar a LLM sem contexto
resultados_baseline = []

for i, teste in enumerate(testes, 1):
    resposta, latencia = gerar_resposta(
        teste["pergunta"],
        contexto=None
    )

    try:
        json.loads(resposta)
        json_valido = True
    except json.JSONDecodeError:
        json_valido = False

    resultados_baseline.append({
        "pergunta": teste["pergunta"],
        "gabarito": teste["gabarito"],
        "resposta_original": resposta,
        "json_valido": json_valido,
        "latencia_segundos": latencia,
        "deve_recusar": teste["deve_recusar"]
    })

    print(f"Pergunta {i}/15 concluída")

# Salvar resultados
with open(
    "avaliacao/baseline_sem_contexto.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        resultados_baseline,
        arquivo,
        ensure_ascii=False,
        indent=2
    )

validos = sum(
    r["json_valido"] for r in resultados_baseline
)

print("\n===== RESULTADO BASELINE =====")
print("Perguntas executadas:", len(resultados_baseline))
print("JSONs originalmente válidos:", validos, "/15")
print("Arquivo salvo: avaliacao/baseline_sem_contexto.json")

print("\nEXEMPLOS DE RESPOSTAS:")
for indice in [0, 2, 10]:
    resultado = resultados_baseline[indice]
    print("\nPergunta:", resultado["pergunta"])
    print("Resposta:", resultado["resposta_original"])
    print("-" * 40)


Pergunta 1/15 concluída
Pergunta 2/15 concluída
Pergunta 3/15 concluída
Pergunta 4/15 concluída
Pergunta 5/15 concluída
Pergunta 6/15 concluída
Pergunta 7/15 concluída
Pergunta 8/15 concluída
Pergunta 9/15 concluída
Pergunta 10/15 concluída
Pergunta 11/15 concluída
Pergunta 12/15 concluída
Pergunta 13/15 concluída
Pergunta 14/15 concluída
Pergunta 15/15 concluída

===== RESULTADO BASELINE =====
Perguntas executadas: 15
JSONs originalmente válidos: 14 /15
Arquivo salvo: avaliacao/baseline_sem_contexto.json

EXEMPLOS DE RESPOSTAS:

Pergunta: Quem deve conservar a calçada em frente ao imóvel?
Resposta: {
  "resposta": "O proprietário ou responsável pelo imóvel deve conservar a calçada em frente ao mesmo.",
  "fontes": [],
  "recusa": false
}
----------------------------------------

Pergunta: Condomínios podem ser responsáveis pelas calçadas?
Resposta: {
  "resposta": "Sim, muitos condomínios têm políticas e regulamentos que incluem a responsabilidade por manter as calçadas limpas e bem c

In [ ]:

import json
import os

LIMIAR_HIBRIDO = 0.26
resultados_rag = []

for i, teste in enumerate(testes, 1):
    pergunta = teste["pergunta"]

    encontrados = buscar_hibrido(pergunta, top_k=3)

    aprovados = [
        t for t in encontrados
        if t["score_hibrido"] >= LIMIAR_HIBRIDO
    ]

    if not aprovados:
        # Recusa pelo filtro, sem executar a LLM
        dados = {
            "resposta": "Não consta na base de conhecimento fornecida.",
            "fontes": [],
            "recusa": True
        }

        resposta = json.dumps(dados, ensure_ascii=False)
        origem = "filtro"
        latencia = 0
        json_llm_valido = None

    else:
        contexto = "\n\n".join([
            f"ID: {t['id']}\n"
            f"Fonte: {t['fonte']}\n"
            f"Texto: {t['texto']}"
            for t in aprovados
        ])

        resposta, latencia = gerar_resposta(
            pergunta, contexto
        )

        origem = "llm"

        try:
            dados = json.loads(resposta)
            json_llm_valido = (
                isinstance(dados, dict)
                and isinstance(dados.get("resposta"), str)
                and isinstance(dados.get("fontes"), list)
                and all(
                    isinstance(f, str)
                    for f in dados["fontes"]
                )
                and isinstance(dados.get("recusa"), bool)
            )
        except json.JSONDecodeError:
            dados = None
            json_llm_valido = False

    ids_permitidos = {t["id"] for t in aprovados}

    fontes_validas = (
        isinstance(dados, dict)
        and isinstance(dados.get("fontes"), list)
        and all(
            f in ids_permitidos
            for f in dados["fontes"]
        )
    )

    resultados_rag.append({
        "pergunta": pergunta,
        "gabarito": teste["gabarito"],
        "deve_recusar": teste["deve_recusar"],
        "resposta_original": resposta,
        "origem": origem,
        "json_llm_valido": json_llm_valido,
        "fontes_validas": fontes_validas,
        "latencia_segundos": latencia,
        "fontes_recuperadas": list(ids_permitidos),
        "recusa_final": (
            dados.get("recusa")
            if isinstance(dados, dict)
            else None
        )
    })

    print(f"Pergunta {i}/15 concluída")

os.makedirs("avaliacao", exist_ok=True)

with open(
    "avaliacao/resultados_rag.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        resultados_rag,
        arquivo,
        indent=2,
        ensure_ascii=False
    )

gerados = [
    r for r in resultados_rag
    if r["origem"] == "llm"
]

validos = sum(
    r["json_llm_valido"] is True
    for r in gerados
)

recusas_corretas = sum(
    r["recusa_final"] is True
    for r in resultados_rag
    if r["deve_recusar"]
)

print("\n===== RESULTADO RAG =====")
print("Perguntas:", len(resultados_rag))
print("Respostas geradas pela LLM:", len(gerados))
print("Recusas feitas pelo filtro:", 15 - len(gerados))
print("JSONs válidos da LLM:", validos, "/", len(gerados))
print("Recusas corretas do sistema:", recusas_corretas, "/5")

print("\nEXEMPLOS:")
for indice in [0, 2, 10]:
    r = resultados_rag[indice]
    print("\nPergunta:", r["pergunta"])
    print("Origem:", r["origem"])
    print("Resposta:", r["resposta_original"])
    print("-" * 40)


Pergunta 1/15 concluída
Pergunta 2/15 concluída
Pergunta 3/15 concluída
Pergunta 4/15 concluída
Pergunta 5/15 concluída
Pergunta 6/15 concluída
Pergunta 7/15 concluída
Pergunta 8/15 concluída
Pergunta 9/15 concluída
Pergunta 10/15 concluída
Pergunta 11/15 concluída
Pergunta 12/15 concluída
Pergunta 13/15 concluída
Pergunta 14/15 concluída
Pergunta 15/15 concluída

===== RESULTADO RAG =====
Perguntas: 15
Respostas geradas pela LLM: 9
Recusas feitas pelo filtro: 6
JSONs válidos da LLM: 4 / 9
Recusas corretas do sistema: 5 /5

EXEMPLOS:

Pergunta: Quem deve conservar a calçada em frente ao imóvel?
Origem: llm
Resposta: ```json
{
  "resposta": "Em regra, os responsáveis pelos imóveis devem executar, manter e conservar as calçadas correspondentes à frente do imóvel, observadas as exceções legais.",
  "fontes": ["lei_15442_art7"],
  "recusa": false
}
```
----------------------------------------

Pergunta: Condomínios podem ser responsáveis pelas calçadas?
Origem: filtro
Resposta: {"resposta"

In [ ]:

import json
import re

resultados_normalizados = []

for r in resultados_rag:
    texto = r["resposta_original"].strip()

    # Identificar JSON dentro de Markdown
    texto_limpo = re.sub(
        r"^```(?:json)?\s*", "", texto,
        flags=re.IGNORECASE
    )
    texto_limpo = re.sub(
        r"\s*```$", "", texto_limpo
    ).strip()

    try:
        dados = json.loads(texto_limpo)

        estrutura_valida = (
            isinstance(dados, dict)
            and isinstance(dados.get("resposta"), str)
            and isinstance(dados.get("fontes"), list)
            and isinstance(dados.get("recusa"), bool)
        )

        fontes_validas = (
            estrutura_valida
            and all(
                isinstance(f, str)
                and f in r["fontes_recuperadas"]
                for f in dados["fontes"]
            )
        )

        resposta_valida = (
            estrutura_valida
            and fontes_validas
            and (
                dados["recusa"]
                or len(dados["fontes"]) > 0
            )
        )

    except (json.JSONDecodeError, TypeError):
        resposta_valida = False

    resultados_normalizados.append({
        **r,
        "json_normalizado_valido": resposta_valida
    })

validos = sum(
    r["json_normalizado_valido"]
    for r in resultados_normalizados
)

print("===== VALIDAÇÃO DO SISTEMA =====")
print("Respostas válidas após tratamento:", validos, "/15")
print("JSONs originalmente válidos da LLM: 4/9")

print("\nFALHAS QUE AINDA EXISTEM:")
for r in resultados_normalizados:
    if not r["json_normalizado_valido"]:
        print("-", r["pergunta"])

print("\nRECUSAS INDEVIDAS DO FILTRO:")
for r in resultados_normalizados:
    if r["origem"] == "filtro" and not r["deve_recusar"]:
        print("-", r["pergunta"])

with open(
    "avaliacao/resultados_rag_normalizados.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        resultados_normalizados,
        arquivo,
        ensure_ascii=False,
        indent=2
    )

print("\nResultados salvos!")


===== VALIDAÇÃO DO SISTEMA =====
Respostas válidas após tratamento: 13 /15
JSONs originalmente válidos da LLM: 4/9

FALHAS QUE AINDA EXISTEM:
- Buracos podem caracterizar calçada mal conservada?
- Desníveis indevidos prejudicam a conservação da calçada?

RECUSAS INDEVIDAS DO FILTRO:
- Condomínios podem ser responsáveis pelas calçadas?

Resultados salvos!


In [ ]:

print("===== DIAGNÓSTICO DOS ERROS =====")

for r in resultados_normalizados:
    if not r["json_normalizado_valido"]:
        print("\nPERGUNTA:", r["pergunta"])
        print("Origem:", r["origem"])
        print("Fontes recuperadas:", r["fontes_recuperadas"])
        print("Resposta original:")
        print(r["resposta_original"])
        print("-" * 50)


===== DIAGNÓSTICO DOS ERROS =====

PERGUNTA: Buracos podem caracterizar calçada mal conservada?
Origem: llm
Fontes recuperadas: ['decreto_59671_art34']
Resposta original:
{
  "resposta": "Sim, buracos podem caracterizar uma calçada em mau estado de conservação.",
  "fontes": ["Decreto 59.671/2020 - Art. 34"],
  "recusa": false
}
--------------------------------------------------

PERGUNTA: Desníveis indevidos prejudicam a conservação da calçada?
Origem: llm
Fontes recuperadas: ['decreto_59671_art34']
Resposta original:
```json
{
  "resposta": "Sim, desníveis indevidos podem prejudicar a conservação da calçada.",
  "fontes": ["Decreto 59.671/2020 - Art. 34"],
  "recusa": false
}
```
--------------------------------------------------


In [ ]:

import json
import re

def validar_resposta_com_fontes(resultado):
    texto = resultado["resposta_original"].strip()

    # Remover formatação Markdown
    texto = re.sub(
        r"^```(?:json)?\s*", "",
        texto,
        flags=re.IGNORECASE
    )
    texto = re.sub(r"\s*```$", "", texto).strip()

    try:
        dados = json.loads(texto)

        if not (
            isinstance(dados, dict)
            and isinstance(dados.get("resposta"), str)
            and isinstance(dados.get("fontes"), list)
            and isinstance(dados.get("recusa"), bool)
        ):
            return False, None

        # Aceitar somente fontes recuperadas
        fontes_permitidas = {}

        for fonte_id in resultado["fontes_recuperadas"]:
            for documento in base_conhecimento:
                if documento["id"] == fonte_id:
                    fontes_permitidas[fonte_id] = fonte_id
                    fontes_permitidas[documento["fonte"]] = fonte_id

        fontes_normalizadas = []

        for fonte in dados["fontes"]:
            if not isinstance(fonte, str):
                return False, None

            if fonte not in fontes_permitidas:
                return False, None

            fontes_normalizadas.append(fontes_permitidas[fonte])

        dados["fontes"] = list(dict.fromkeys(fontes_normalizadas))

        if not dados["recusa"] and not dados["fontes"]:
            return False, None

        return True, dados

    except (json.JSONDecodeError, TypeError, ValueError):
        return False, None


# Reavaliar as saídas originais do RAG
validos = 0

for resultado in resultados_rag:
    valido, dados = validar_resposta_com_fontes(resultado)

    if valido:
        validos += 1
    else:
        print("Falha:", resultado["pergunta"])

print("\n===== RESULTADO =====")
print(f"Saídas válidas após normalização: {validos}/15")
print("JSONs originalmente válidos da LLM: 4/9")

print("\nRecusas indevidas:")
for r in resultados_rag:
    if r["origem"] == "filtro" and not r["deve_recusar"]:
        print("-", r["pergunta"])



===== RESULTADO =====
Saídas válidas após normalização: 15/15
JSONs originalmente válidos da LLM: 4/9

Recusas indevidas:
- Condomínios podem ser responsáveis pelas calçadas?


In [ ]:

import re
import unicodedata
from collections import Counter

def extrair_termos(texto):
    texto = unicodedata.normalize("NFKD", texto.lower())
    texto = "".join(
        c for c in texto if not unicodedata.combining(c)
    )
    return set(re.findall(r"[a-z]{5,}", texto))

# Identificar termos específicos dos documentos
termos_documentos = [
    extrair_termos(doc["texto"])
    for doc in base_conhecimento
]

frequencia = Counter(
    termo
    for termos in termos_documentos
    for termo in termos
)

def busca_agente(pergunta):
    # Ferramenta 1: busca híbrida convencional
    resultados = buscar_hibrido(pergunta, top_k=3)

    aprovados = [
        r for r in resultados
        if r["score_hibrido"] >= 0.26
    ]

    if aprovados:
        return aprovados, "busca_hibrida"

    # Ferramenta 2: recuperar documentos por termos específicos
    termos_pergunta = extrair_termos(pergunta)

    candidatos = []

    for i, doc in enumerate(base_conhecimento):
        termos_raros = {
            t for t in termos_pergunta
            if frequencia[t] == 1
            and t in termos_documentos[i]
        }

        if termos_raros:
            candidatos.append((
                len(termos_raros),
                doc,
                termos_raros
            ))

    if candidatos:
        candidatos.sort(key=lambda x: x[0], reverse=True)
        return [
            item[1] for item in candidatos[:3]
        ], "busca_especifica"

    return [], "sem_evidencias"


# Testar as duas situações
perguntas_agente = [
    "Condomínios podem ser responsáveis pelas calçadas?",
    "Quantas reclamações sobre calçadas foram registradas ontem?"
]

for pergunta in perguntas_agente:
    encontrados, ferramenta = busca_agente(pergunta)

    print("\nPERGUNTA:", pergunta)
    print("Ferramenta utilizada:", ferramenta)
    print("Fontes encontradas:",
          [d["id"] for d in encontrados])
    print("-" * 50)



PERGUNTA: Condomínios podem ser responsáveis pelas calçadas?
Ferramenta utilizada: busca_especifica
Fontes encontradas: ['lei_15442_art10']
--------------------------------------------------

PERGUNTA: Quantas reclamações sobre calçadas foram registradas ontem?
Ferramenta utilizada: sem_evidencias
Fontes encontradas: []
--------------------------------------------------


In [ ]:

import json
import os
import time

resultados_agente = []

for i, teste in enumerate(testes, 1):
    pergunta = teste["pergunta"]
    inicio = time.perf_counter()

    # Agente escolhe a ferramenta de busca
    documentos, ferramenta = busca_agente(pergunta)
    ids = [d["id"] for d in documentos]

    if not documentos:
        # Recusa automática por falta de evidências
        dados = {
            "resposta": "Não consta na base de conhecimento fornecida.",
            "fontes": [],
            "recusa": True
        }

        resposta = json.dumps(dados, ensure_ascii=False)
        origem = "filtro"
        json_original_valido = None
        latencia_llm = 0
        validacao_ok = True

    else:
        # Preparar documentos encontrados
        contexto = "\n\n".join(
            f"ID: {d['id']}\n"
            f"Fonte: {d['fonte']}\n"
            f"Texto: {d['texto']}"
            for d in documentos
        )

        # Chamar a LLM
        resposta, latencia_llm = gerar_resposta(
            pergunta, contexto
        )

        origem = "llm"

        try:
            json.loads(resposta)
            json_original_valido = True
        except json.JSONDecodeError:
            json_original_valido = False

        # Validar e normalizar as fontes
        validacao_ok, dados = validar_resposta_com_fontes({
            "resposta_original": resposta,
            "fontes_recuperadas": ids
        })

        if not validacao_ok:
            dados = {
                "resposta": "Não foi possível validar a resposta.",
                "fontes": [],
                "recusa": True
            }

    tempo_total = time.perf_counter() - inicio

    resultados_agente.append({
        "pergunta": pergunta,
        "gabarito": teste["gabarito"],
        "deve_recusar": teste["deve_recusar"],
        "ferramenta": ferramenta,
        "origem": origem,
        "fontes_recuperadas": ids,
        "resposta_original": resposta,
        "resposta_final": dados,
        "json_original_valido": json_original_valido,
        "validacao_ok": validacao_ok,
        "latencia_llm": latencia_llm,
        "latencia_total": round(tempo_total, 2)
    })

    print(f"Pergunta {i}/15 concluída")

os.makedirs("avaliacao", exist_ok=True)

with open(
    "avaliacao/resultados_agente.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        resultados_agente,
        arquivo,
        ensure_ascii=False,
        indent=2
    )

geradas = [
    r for r in resultados_agente
    if r["origem"] == "llm"
]

jsons_validos = sum(
    r["json_original_valido"] is True
    for r in geradas
)

recusas_corretas = sum(
    r["deve_recusar"]
    and r["resposta_final"]["recusa"]
    for r in resultados_agente
)

recusas_incorretas = sum(
    not r["deve_recusar"]
    and r["resposta_final"]["recusa"]
    for r in resultados_agente
)

print("\n===== RESULTADO DO AGENTE =====")
print("Perguntas:", len(resultados_agente))
print("Respostas geradas pela LLM:", len(geradas))
print("JSONs originais válidos:", jsons_validos, "/", len(geradas))
print("Recusas corretas:", recusas_corretas, "/5")
print("Recusas indevidas:", recusas_incorretas)
print("Arquivo salvo: avaliacao/resultados_agente.json")

print("\n===== EXEMPLOS =====")

for indice in [2, 8, 10]:
    r = resultados_agente[indice]

    print("\nPergunta:", r["pergunta"])
    print("Ferramenta:", r["ferramenta"])
    print("Resposta final:")
    print(json.dumps(
        r["resposta_final"],
        indent=2,
        ensure_ascii=False
    ))
    print("-" * 40)


Pergunta 1/15 concluída
Pergunta 2/15 concluída
Pergunta 3/15 concluída
Pergunta 4/15 concluída
Pergunta 5/15 concluída
Pergunta 6/15 concluída
Pergunta 7/15 concluída
Pergunta 8/15 concluída
Pergunta 9/15 concluída
Pergunta 10/15 concluída
Pergunta 11/15 concluída
Pergunta 12/15 concluída
Pergunta 13/15 concluída
Pergunta 14/15 concluída
Pergunta 15/15 concluída

===== RESULTADO DO AGENTE =====
Perguntas: 15
Respostas geradas pela LLM: 10
JSONs originais válidos: 5 / 10
Recusas corretas: 5 /5
Recusas indevidas: 0
Arquivo salvo: avaliacao/resultados_agente.json

===== EXEMPLOS =====

Pergunta: Condomínios podem ser responsáveis pelas calçadas?
Ferramenta: busca_especifica
Resposta final:
{
  "resposta": "Sim, conforme a Lei 15.442/2011, artigo 10, os condomínios são responsáveis por manter as calçadas.",
  "fontes": [
    "lei_15442_art10"
  ],
  "recusa": false
}
----------------------------------------

Pergunta: Buracos podem caracterizar calçada mal conservada?
Ferramenta: busca_hi

In [1]:

import json
import os

def interpretar_baseline(r):
    try:
        dados = json.loads(r["resposta_original"])
        return dados if isinstance(dados, dict) else None
    except (json.JSONDecodeError, TypeError):
        return None

def interpretar_rag(r):
    valido, dados = validar_resposta_com_fontes(r)
    return dados if valido else None

def interpretar_agente(r):
    return r["resposta_final"]

configuracoes = [
    ("Sem contexto", resultados_baseline, interpretar_baseline),
    ("RAG simples", resultados_rag, interpretar_rag),
    ("RAG + agente", resultados_agente, interpretar_agente)
]

placar = []

for nome, resultados, interpretar in configuracoes:
    recusas_corretas = 0
    recusas_indevidas = 0
    citacoes_corretas = 0
    json_originais = 0
    chamadas_llm = 0

    for teste, r in zip(testes, resultados):
        assert teste["pergunta"] == r["pergunta"]

        if nome == "Sem contexto":
            chamadas_llm += 1
            json_originais += int(r["json_valido"])
        elif r["origem"] == "llm":
            chamadas_llm += 1
            chave = (
                "json_llm_valido"
                if nome == "RAG simples"
                else "json_original_valido"
            )
            json_originais += int(r[chave] is True)

        dados = interpretar(r)

        if dados is None:
            continue

        recusa = dados.get("recusa")
        fontes = dados.get("fontes", [])

        if teste["deve_recusar"]:
            if recusa is True:
                recusas_corretas += 1
        else:
            if recusa is True:
                recusas_indevidas += 1

            if isinstance(fontes, list) and any(
                fonte in fontes
                for fonte in teste["fontes_esperadas"]
            ):
                citacoes_corretas += 1

    placar.append({
        "configuracao": nome,
        "json_originais_validos": json_originais,
        "chamadas_llm": chamadas_llm,
        "recusas_corretas": recusas_corretas,
        "recusas_indevidas": recusas_indevidas,
        "respostas_com_fonte_esperada": citacoes_corretas
    })

print("===== PLACAR COMPARATIVO =====")

for p in placar:
    print("\nCONFIGURAÇÃO:", p["configuracao"])
    print(
        "JSONs originais:",
        f'{p["json_originais_validos"]}/{p["chamadas_llm"]}'
    )
    print("Recusas corretas:", p["recusas_corretas"], "/5")
    print("Recusas indevidas:", p["recusas_indevidas"])
    print(
        "Respostas com fonte esperada:",
        p["respostas_com_fonte_esperada"], "/10"
    )

os.makedirs("avaliacao", exist_ok=True)

with open(
    "avaliacao/placar_preliminar.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(
        placar, arquivo,
        indent=2,
        ensure_ascii=False
    )

print("\nPlacar salvo com sucesso!")


NameError: name 'resultados_baseline' is not defined

In [2]:

import os

arquivos = [
    "avaliacao/testes.json",
    "avaliacao/baseline_sem_contexto.json",
    "avaliacao/resultados_rag.json",
    "avaliacao/resultados_agente.json",
    "base_conhecimento.json"
]

for arquivo in arquivos:
    if os.path.exists(arquivo):
        print("OK:", arquivo)
    else:
        print("NÃO ENCONTRADO:", arquivo)


NÃO ENCONTRADO: avaliacao/testes.json
NÃO ENCONTRADO: avaliacao/baseline_sem_contexto.json
NÃO ENCONTRADO: avaliacao/resultados_rag.json
NÃO ENCONTRADO: avaliacao/resultados_agente.json
NÃO ENCONTRADO: base_conhecimento.json
